In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp

### Lấy dữ liệu giá cổ phiếu

In [ ]:
# Giả sử chúng ta lấy dữ liệu từ Yahoo Finance
import yfinance as yf

tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
data = yf.download(tickers, start="2020-01-01", end="2023-01-01")['Close']
returns = data.pct_change().dropna()


### Tính lợi nhuận kỳ vọng và ma trận hiệp phương sai

In [3]:
mu = returns.mean()  # Lợi nhuận kỳ vọng
cov_matrix = returns.cov()  # Ma trận hiệp phương sai

### Xây dựng mô hình tối ưu

In [4]:
num_assets = len(tickers)
w = cp.Variable(num_assets)  # Trọng số danh mục

risk = cp.quad_form(w, cov_matrix)  # Rủi ro (phương sai danh mục)
ret = cp.matmul(mu.values, w)  # Chuyển đổi mu thành numpy array  # Lợi nhuận kỳ vọng danh mục

prob = cp.Problem(cp.Minimize(risk),  # Mục tiêu: tối thiểu rủi ro
                  [cp.sum(w) == 1, w >= 0])  # Ràng buộc: tổng trọng số = 1, không bán khống

prob.solve()
optimal_weights = w.value  # Trọng số tối ưu


### Vẽ Biên hiệu quả (Efficient Frontier)

In [ ]:
def efficient_frontier(mu, cov_matrix, num_portfolios=100):
    results = []
    for _ in range(num_portfolios):
        w = np.random.dirichlet(np.ones(len(mu)))
        ret = np.dot(w, mu)
        risk = np.sqrt(np.dot(w.T, np.dot(cov_matrix, w)))
        results.append([risk, ret])
    results = np.array(results)
    plt.scatter(results[:, 0], results[:, 1], alpha=0.5, label='Random Portfolios')
    plt.xlabel('Rủi ro')
    plt.ylabel('Lợi nhuận kỳ vọng')
    plt.legend()
    plt.show()

efficient_frontier(mu, cov_matrix)
